<a href="https://colab.research.google.com/github/zoetsouroufli/slp-lab/blob/main/%CE%91%CE%BD%CF%84%CE%AF%CE%B3%CF%81%CE%B1%CF%86%CE%BF_Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers evaluate datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/lab3

/content/drive/MyDrive/lab3


In [ ]:
import numpy as np
import evaluate
from datasets import Dataset
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from utils.load_datasets import load_MR, load_Semeval2017A


DATASET = 'MR'  # 'MR' or 'Semeval2017A'
PRETRAINED_MODEL = 'bert-base-cased'


metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


def tokenize_function(examples):
    #return tokenizer(examples["text"], padding="max_length", truncation=True)
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)


def prepare_dataset(X, y):
    texts, labels = [], []
    for text, label in zip(X, y):
        texts.append(text)
        labels.append(label)

    return Dataset.from_dict({'text': texts, 'label': labels})


if __name__ == '__main__':

    # load the raw data
    if DATASET == "Semeval2017A":
        X_train, y_train, X_test, y_test = load_Semeval2017A()
    elif DATASET == "MR":
        X_train, y_train, X_test, y_test = load_MR()
    else:
        raise ValueError("Invalid dataset")

    # encode labels
    le = LabelEncoder()
    le.fit(list(set(y_train)))
    y_train = le.transform(y_train)
    y_test = le.transform(y_test)
    n_classes = len(list(le.classes_))

    # prepare datasets
    train_set = prepare_dataset(X_train, y_train)
    test_set = prepare_dataset(X_test, y_test)

    # define model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED_MODEL, num_labels=n_classes)

    # tokenize datasets
    tokenized_train_set = train_set.map(tokenize_function)
    tokenized_test_set = test_set.map(tokenize_function)

    # TODO: Main-lab-Q7 - remove this section once you are ready to execute on a GPU
    #  create a smaller subset of the dataset
    n_samples = 40
    small_train_dataset = tokenized_train_set.shuffle(
        seed=42).select(range(n_samples))
    small_eval_dataset = tokenized_test_set.shuffle(
        seed=42).select(range(n_samples))

    # TODO: Main-lab-Q7 - customize hyperparameters once you are ready to execute on a GPU
    # training setup
    args = TrainingArguments(
        output_dir="output",
        eval_strategy="epoch",
        num_train_epochs=3,
        per_device_train_batch_size=16
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train_set,
        eval_dataset=tokenized_test_set,
        compute_metrics=compute_metrics,
    )

    # train
    #trained_model = trainer.train()

    print("Ξεκινάει η εκπαίδευση...")
    trainer.train()

    # ---------------------------------------------------------
    # ΠΡΟΣΘΗΚΗ: Για να δεις τα τελικά αποτελέσματα στο Test Set
    # ---------------------------------------------------------
    print("\n--- Τελική Αξιολόγηση στο Test Set ---")
    results = trainer.evaluate(tokenized_test_set)
    print(results)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Τελική Αξιολόγηση στο Test Set ---


{'eval_loss': 0.9029703140258789, 'eval_accuracy': 0.8595166163141994, 'eval_runtime': 21.6685, 'eval_samples_per_second': 30.551, 'eval_steps_per_second': 3.83, 'epoch': 4.0}


In [ ]:
import numpy as np
import evaluate
from datasets import Dataset
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from utils.load_datasets import load_MR, load_Semeval2017A


DATASET = 'sEM'  # 'MR' or 'Semeval2017A'
PRETRAINED_MODEL = 'bert-base-cased'


metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


def tokenize_function(examples):
    #return tokenizer(examples["text"], padding="max_length", truncation=True)
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)


def prepare_dataset(X, y):
    texts, labels = [], []
    for text, label in zip(X, y):
        texts.append(text)
        labels.append(label)

    return Dataset.from_dict({'text': texts, 'label': labels})


if __name__ == '__main__':

    # load the raw data
    if DATASET == "Semeval2017A":
        X_train, y_train, X_test, y_test = load_Semeval2017A()
    elif DATASET == "MR":
        X_train, y_train, X_test, y_test = load_MR()
    else:
        raise ValueError("Invalid dataset")

    # encode labels
    le = LabelEncoder()
    le.fit(list(set(y_train)))
    y_train = le.transform(y_train)
    y_test = le.transform(y_test)
    n_classes = len(list(le.classes_))

    # prepare datasets
    train_set = prepare_dataset(X_train, y_train)
    test_set = prepare_dataset(X_test, y_test)

    # define model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED_MODEL, num_labels=n_classes)

    # tokenize datasets
    tokenized_train_set = train_set.map(tokenize_function)
    tokenized_test_set = test_set.map(tokenize_function)

    # TODO: Main-lab-Q7 - remove this section once you are ready to execute on a GPU
    #  create a smaller subset of the dataset
    n_samples = 40
    small_train_dataset = tokenized_train_set.shuffle(
        seed=42).select(range(n_samples))
    small_eval_dataset = tokenized_test_set.shuffle(
        seed=42).select(range(n_samples))

    # TODO: Main-lab-Q7 - customize hyperparameters once you are ready to execute on a GPU
    # training setup
    args = TrainingArguments(
        output_dir="output",
        eval_strategy="epoch",
        num_train_epochs=3,
        per_device_train_batch_size=16
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train_set,
        eval_dataset=tokenized_test_set,
        compute_metrics=compute_metrics,
    )

    # train
    #trained_model = trainer.train()

    print("Ξεκινάει η εκπαίδευση...")
    trainer.train()

    # ---------------------------------------------------------
    # ΠΡΟΣΘΗΚΗ: Για να δεις τα τελικά αποτελέσματα στο Test Set
    # ---------------------------------------------------------
    print("\n--- Τελική Αξιολόγηση στο Test Set ---")
    results = trainer.evaluate(tokenized_test_set)
    print(results)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Τελική Αξιολόγηση στο Test Set ---


{'eval_loss': 1.1173758506774902, 'eval_accuracy': 0.6842233800065125, 'eval_runtime': 91.5312, 'eval_samples_per_second': 134.206, 'eval_steps_per_second': 16.781, 'epoch': 3.0}


In [ ]:
import numpy as np
import evaluate
from datasets import Dataset
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from utils.load_datasets import load_MR, load_Semeval2017A


DATASET = 'MR'  # 'MR' or 'Semeval2017A'
PRETRAINED_MODEL = 'roberta-base'


metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


def tokenize_function(examples):
    #return tokenizer(examples["text"], padding="max_length", truncation=True)
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)


def prepare_dataset(X, y):
    texts, labels = [], []
    for text, label in zip(X, y):
        texts.append(text)
        labels.append(label)

    return Dataset.from_dict({'text': texts, 'label': labels})


if __name__ == '__main__':

    # load the raw data
    if DATASET == "Semeval2017A":
        X_train, y_train, X_test, y_test = load_Semeval2017A()
    elif DATASET == "MR":
        X_train, y_train, X_test, y_test = load_MR()
    else:
        raise ValueError("Invalid dataset")

    # encode labels
    le = LabelEncoder()
    le.fit(list(set(y_train)))
    y_train = le.transform(y_train)
    y_test = le.transform(y_test)
    n_classes = len(list(le.classes_))

    # prepare datasets
    train_set = prepare_dataset(X_train, y_train)
    test_set = prepare_dataset(X_test, y_test)

    # define model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED_MODEL, num_labels=n_classes)

    # tokenize datasets
    tokenized_train_set = train_set.map(tokenize_function)
    tokenized_test_set = test_set.map(tokenize_function)

    # TODO: Main-lab-Q7 - remove this section once you are ready to execute on a GPU
    #  create a smaller subset of the dataset
    n_samples = 40
    small_train_dataset = tokenized_train_set.shuffle(
        seed=42).select(range(n_samples))
    small_eval_dataset = tokenized_test_set.shuffle(
        seed=42).select(range(n_samples))

    # TODO: Main-lab-Q7 - customize hyperparameters once you are ready to execute on a GPU
    # training setup
    args = TrainingArguments(
        output_dir="output",
        eval_strategy="epoch",
        num_train_epochs=3,
        per_device_train_batch_size=16
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train_set,
        eval_dataset=tokenized_test_set,
        compute_metrics=compute_metrics,
    )

    # train
    #trained_model = trainer.train()

    print("Ξεκινάει η εκπαίδευση...")
    trainer.train()

    # ---------------------------------------------------------
    # ΠΡΟΣΘΗΚΗ: Για να δεις τα τελικά αποτελέσματα στο Test Set
    # ---------------------------------------------------------
    print("\n--- Τελική Αξιολόγηση στο Test Set ---")
    results = trainer.evaluate(tokenized_test_set)
    print(results)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Τελική Αξιολόγηση στο Test Set ---


{'eval_loss': 0.5020455718040466, 'eval_accuracy': 0.8700906344410876, 'eval_runtime': 4.5901, 'eval_samples_per_second': 144.224, 'eval_steps_per_second': 18.082, 'epoch': 3.0}


In [ ]:
import numpy as np
import evaluate
from datasets import Dataset
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from utils.load_datasets import load_MR, load_Semeval2017A


DATASET = 'Semeval2017A'  # 'MR' or 'Semeval2017A'
PRETRAINED_MODEL = 'roberta-base'


metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


def tokenize_function(examples):
    #return tokenizer(examples["text"], padding="max_length", truncation=True)
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)


def prepare_dataset(X, y):
    texts, labels = [], []
    for text, label in zip(X, y):
        texts.append(text)
        labels.append(label)

    return Dataset.from_dict({'text': texts, 'label': labels})


if __name__ == '__main__':

    # load the raw data
    if DATASET == "Semeval2017A":
        X_train, y_train, X_test, y_test = load_Semeval2017A()
    elif DATASET == "MR":
        X_train, y_train, X_test, y_test = load_MR()
    else:
        raise ValueError("Invalid dataset")

    # encode labels
    le = LabelEncoder()
    le.fit(list(set(y_train)))
    y_train = le.transform(y_train)
    y_test = le.transform(y_test)
    n_classes = len(list(le.classes_))

    # prepare datasets
    train_set = prepare_dataset(X_train, y_train)
    test_set = prepare_dataset(X_test, y_test)

    # define model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED_MODEL, num_labels=n_classes)

    # tokenize datasets
    tokenized_train_set = train_set.map(tokenize_function)
    tokenized_test_set = test_set.map(tokenize_function)

    # TODO: Main-lab-Q7 - remove this section once you are ready to execute on a GPU
    #  create a smaller subset of the dataset
    n_samples = 40
    small_train_dataset = tokenized_train_set.shuffle(
        seed=42).select(range(n_samples))
    small_eval_dataset = tokenized_test_set.shuffle(
        seed=42).select(range(n_samples))

    # TODO: Main-lab-Q7 - customize hyperparameters once you are ready to execute on a GPU
    # training setup
    args = TrainingArguments(
        output_dir="output",
        eval_strategy="epoch",
        num_train_epochs=3,
        per_device_train_batch_size=16
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train_set,
        eval_dataset=tokenized_test_set,
        compute_metrics=compute_metrics,
    )

    # train
    #trained_model = trainer.train()

    print("Ξεκινάει η εκπαίδευση...")
    trainer.train()

    # ---------------------------------------------------------
    # ΠΡΟΣΘΗΚΗ: Για να δεις τα τελικά αποτελέσματα στο Test Set
    # ---------------------------------------------------------
    print("\n--- Τελική Αξιολόγηση στο Test Set ---")
    results = trainer.evaluate(tokenized_test_set)
    print(results)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/49570 [00:00<?, ? examples/s]

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

Ξεκινάει η εκπαίδευση...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Τελική Αξιολόγηση στο Test Set ---


{'eval_loss': 0.7810913324356079, 'eval_accuracy': 0.7017258222077499, 'eval_runtime': 86.72, 'eval_samples_per_second': 141.651, 'eval_steps_per_second': 17.712, 'epoch': 3.0}
